<a href="https://colab.research.google.com/github/ismailapan/time_series_analysis_and_ML/blob/main/cv_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import userdata

data_path = userdata.get('DATA_PATH')

rmsse_df = pd.read_csv(data_path)

In [ ]:
import numpy as np

def rmsse(actual, pred, train_actual):
    """
    actual, pred: real and predict values of val and test periods
    train_actual: weekly real sales series of this product in train set
    """
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    train_actual = np.asarray(np.diff(train_actual), dtype=float)

    numerator = np.mean((actual - pred) ** 2)
    denominator = np.mean(np.diff(train_actual) ** 2)

    if denominator == 0:
        return np.nan

    return np.sqrt(numerator / denominator)


In [ ]:
segment_result = pd.DataFrame([
    (1, "LowData",      5076, 0.071, 0.071),
    (1, "Intermittent", 2368, 0.388, 0.389),
    (1, "Lumpy",        1358, 0.563, 0.568),
    (1, "Erratic",       849, 0.518, 0.523),
    (1, "Smooth",         96, 0.583, 0.583),

    (2, "LowData",      3541, 0.071, 0.071),
    (2, "Intermittent", 1673, 0.432, 0.433),
    (2, "Lumpy",         944, 0.539, 0.537),
    (2, "Erratic",       554, 0.567, 0.560),
    (2, "Smooth",         74, 0.432, 0.473),

    (3, "LowData",      5641, 0.076, 0.076),
    (3, "Intermittent", 2609, 0.458, 0.458),
    (3, "Lumpy",        1492, 0.623, 0.623),
    (3, "Erratic",       859, 0.584, 0.583),
    (3, "Smooth",        124, 0.565, 0.565),
], columns=["fold", "segment", "n", "wape_win_rate", "teknorot_win_rate"])

In [ ]:
summary = segment_result.groupby('segment').agg(
    n_total = ("n", "sum"),
    wape_win_mean = ("wape_win_rate", "mean"),
    wape_win_std=("wape_win_rate", "std"),
    teknorot_win_mean=("teknorot_win_rate", "mean"),
    teknorot_win_std=("teknorot_win_rate", "std"),
).reset_index()

summary['decision'] = summary['teknorot_win_mean'].apply(
    lambda x: 'model' if x > 0.50 else 'baseline'
)

In [ ]:
rmsse_summary = rmsse_df.groupby('sbc_category').agg(
    rmsse_model_mean = ('rmsse_model', 'mean'),
    rmsse_baseline_mean = ('rmsse_baseline', 'mean')
).reset_index().rename(columns={'sbc_category':'segment'})

final = summary.merge(rmsse_summary, on='segment')

,segment,n_total,wape_win_mean,wape_win_std,teknorot_win_mean,teknorot_win_std,decision,rmsse_model_mean,rmsse_baseline_mean
0,Erratic,2262,0.556333,0.034269,0.555333,0.030271,model,0.366911,0.372630
1,Intermittent,6650,0.426000,0.035384,0.426667,0.034933,baseline,0.348078,0.346302
2,LowData,14258,0.072667,0.002887,0.072667,0.002887,baseline,0.237295,0.201843
3,Lumpy,3794,0.575000,0.043267,0.576000,0.043555,model,0.268642,0.276733
4,Smooth,294,0.526667,0.082476,0.540333,0.059003,model,0.536055,0.542762


In [ ]:
GUARDRAIL_PCT = 0.05

def apply_guardrail(row, guardrail_pct=GUARDRAIL_PCT):
    reference = min(row['rmsse_model_mean'], row['rmsse_baseline_mean'])
    chosen_rmsse = row['rmsse_model_mean'] if row['decision'] == 'model' else row['rmsse_baseline_mean']

    if chosen_rmsse > reference * (1 + guardrail_pct):
        safer = 'model' if row['rmsse_model_mean'] < row['rmsse_baseline_mean'] else 'baseline'
        return pd.Series({'final_decision': safer, 'guardrail_override': True})
    return pd.Series({'final_decision': row['decision'], 'guardrail_override': False})

final[['final_decision', 'guardrail_override']] = final.apply(apply_guardrail, axis=1)

print(final[['segment', 'n_total', 'teknorot_win_mean', 'decision',
              'rmsse_model_mean', 'rmsse_baseline_mean', 'final_decision', 'guardrail_override']])


        segment  n_total  teknorot_win_mean  decision  rmsse_model_mean  \
0       Erratic     2262           0.555333     model          0.366911   
1  Intermittent     6650           0.426667  baseline          0.348078   
2       LowData    14258           0.072667  baseline          0.237295   
3         Lumpy     3794           0.576000     model          0.268642   
4        Smooth      294           0.540333     model          0.536055   

   rmsse_baseline_mean final_decision  guardrail_override  
0             0.372630          model               False  
1             0.346302       baseline               False  
2             0.201843       baseline               False  
3             0.276733          model               False  
4             0.542762          model               False  
